# Geospatial ML — Train on Colab & Save to Drive

This notebook trains:
1. **ResNet-50 classifier** on EuroSAT (land-cover classification)
2. **Convolutional Autoencoder** for anomaly detection

Checkpoints are saved to your Google Drive under `MyDrive/geospatial_checkpoints/`.

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/geospatial_checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_OUTPUT}')

## Step 2 — Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/iamvisheshsrivastava/geospatial
%cd geospatial
!git log --oneline -3

## Step 3 — Install dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    rasterio \
    wandb \
    boto3 \
    scikit-learn \
    numpy pandas matplotlib pillow \
    pydantic pydantic-settings \
    tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 4 — Download & extract EuroSAT dataset (~90 MB)

Uses `wget --no-check-certificate` to avoid SSL issues with the DFKI server.

Rather than assuming a fixed ZIP structure (which varies between releases), we extract
everything into a temp folder, then **search the extracted tree for the known class directories**
and move them into place. This works regardless of how many nesting levels the ZIP uses.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]
EUROSAT_DIR = Path('data/eurosat')
EUROSAT_DIR.mkdir(parents=True, exist_ok=True)

# --- Download ---
zip_path = EUROSAT_DIR / 'EuroSAT.zip'
print('Downloading EuroSAT...')
!wget -q --show-progress --no-check-certificate \
    'https://madm.dfki.de/files/sentinel/EuroSAT.zip' \
    -O {zip_path}
print('Download complete.')

# --- Inspect ZIP structure (first 20 entries) so we can see what's inside ---
print('\nZIP contents (first 20 entries):')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist()[:20]:
        print(f'  {name}')

# --- Extract into a temp folder ---
tmp_dir = EUROSAT_DIR / '_tmp_extract'
tmp_dir.mkdir(exist_ok=True)
print('\nExtracting...')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(tmp_dir)

# --- Find class dirs anywhere in the extracted tree and move them ---
# Walk every directory in the extracted tree; if its name matches a class, move it.
found = {}
for dirpath, dirnames, _ in os.walk(tmp_dir):
    for d in dirnames:
        if d in CLASSES and d not in found:
            found[d] = Path(dirpath) / d

print(f'Found {len(found)}/10 class directories in ZIP.')
for cls, src in found.items():
    dest = EUROSAT_DIR / cls
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(src), str(dest))

# --- Clean up ---
shutil.rmtree(tmp_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)

# --- Verify ---
print('\nDataset verification:')
all_ok = True
for cls in CLASSES:
    d = EUROSAT_DIR / cls
    count = len(list(d.glob('*'))) if d.exists() else 0
    status = 'OK' if count > 0 else 'MISSING'
    if count == 0:
        all_ok = False
    print(f'  {status:7s}  {cls} ({count} images)')

if not all_ok:
    raise RuntimeError('Some class directories are missing — check ZIP contents above and report the actual paths shown.')
else:
    print('\nAll 10 classes ready.')

## Step 5 — Train the ResNet-50 classifier

~10 minutes on T4 GPU. Trains for 10 epochs with full fine-tuning.

What this does: fine-tunes a ResNet-50 (pretrained on ImageNet) to classify satellite
patches into 10 land-cover types. The best checkpoint (highest val F1) is saved automatically.

In [ ]:
!python -m src.train \
    --data-root data/eurosat \
    --epochs 10 \
    --batch-size 64 \
    --learning-rate 3e-4 \
    --num-workers 2 \
    --checkpoint-dir checkpoints \
    --wandb-mode disabled

## Step 6 — Train the Anomaly Detector (Autoencoder)

~10 minutes on T4 GPU. Trains on the Forest class only.

What this does: trains a convolutional autoencoder to reconstruct normal forest patches.
At inference time, patches that are hard to reconstruct (high error) are flagged as anomalous
— e.g. deforestation, fire damage, or industrial intrusion.

In [ ]:
!python -m src.anomaly \
    --data-root data/eurosat \
    --normal-classes Forest \
    --epochs 30 \
    --wandb-mode disabled

## Step 7 — Verify checkpoints were created

In [ ]:
import os
for f in ['checkpoints/best_model.pt', 'checkpoints/autoencoder_best.pt']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1_048_576
        print(f'  OK  {f}  ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING  {f} — check training output above for errors')

## Step 8 — Copy checkpoints to Google Drive

Colab sessions are temporary — the checkpoints will be lost when the session ends.
This step copies them to your Google Drive so you can download them afterwards.

In [ ]:
import shutil, os

files_to_save = [
    'checkpoints/best_model.pt',
    'checkpoints/autoencoder_best.pt',
    'checkpoints/training_history.json',
]

for src in files_to_save:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(src))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1_048_576
        print(f'  Saved  {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'  SKIPPED (not found): {src}')

print('\nDone! Download these two files from Google Drive:')
print('  best_model.pt        — ResNet-50 classifier')
print('  autoencoder_best.pt  — Anomaly detector')

## Step 9 — Quick sanity check (optional)

Loads the trained classifier and runs a prediction on a random EuroSAT image.
If this prints a class name and confidence score, the model is working correctly.

In [ ]:
import torch
from pathlib import Path
from src.models.resnet import build_resnet50_classifier
from src.data.preprocessing import preprocess_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('checkpoints/best_model.pt', map_location=device)
class_names = ckpt['class_names']
model = build_resnet50_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

# Grab a random image from the dataset
sample = next(Path('data/eurosat').glob('*/*.jpg'))
tensor = preprocess_image(sample, 224).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze()

conf, idx = probs.max(0)
print(f'Image     : {sample}')
print(f'Predicted : {class_names[idx]} ({conf:.1%} confidence)')
print(f'Val F1    : {ckpt["metrics"]["macro_f1"]:.4f}')